# Условная оптимизация

## Метод штрафных функций

В предыдущей работе нам было необходимо не только решить задачу оптимизации, но и учесть ораничения на возможные решения.

Как вы могли заметить, не все методы оптимизации из тех, которые доступны в пакете SciPy, поддерживают условную оптимизацию, при этом для некоторых задач одни методы более предпочтительны, чем другие, и может так оказаться, что для решения именно вашей задачи тот метод, который поддерживает условную оптимизацию, неприменим, а ограничения все-таки есть.

В этом случае на помощь приходят методы штрафных функций и барьерных поверхностей.

Основная идея у этих методов очень похожа, и заключается в том, что ограничения включаются прямо в оптимизируемую функцию, и делают ее тем более неоптимальной, чем дальше решение заходит в запрещенную область.

Разница между этими методами состоит в том, где находится исходная точка, с которой начинается поиск оптимального решения. В случае метода штрафных функций начальная точка не обязана находиться внутри зоны допустимых с точки зрения ограничений решений, а в случае барьерных функций - должна обязательно. Связано это с тем, что барьерные функции за пределами границы вновь уменьшаются, а штрафные функции проболжают возрастать тем больше, чем дальше за пределы границы вышла текущая точка.

Например, необходимо найти минимум функции $f(x) = (x - 2)^2 + 1$.

Мы знаем, что минимум данной функции находится в точке  $x = 2$, и значение минимизируемой функции в ней равно $1$.

Допустим, что существует ограничение $-\infty < x \le 1$. В этом случае минимум функции будет как раз в точке 1.

Попробуем ввести барьерную функцию. Требования к ней просты: в допустимой зоне ее значение должно быть близким к нулю, чтобы не оказывать влияния на целевую функцию, а в запрещенной зоне ее значение должно увеличиваться.

Введем функцию $g(x) = \frac {1} {1 - x}$. Ее особенность состоит в том, что внутри целевого диапазона она достаточно мала, а при приближении к границе (число 1), начинает возрастать (поскольку $\frac {1} / {1 - 1}$ становится равной бесконечности).

Таким образом, целевая функция теперь приобретает вот такой вид:

$$
f(x) = (x - 2)^2 + 1 + \frac {1} {1 - x}
$$

Казалось бы, все хорошо? На самом деле, нет.

Дело в том, что для данного случая проблема заключается вот в чем: минимум нашей фукнции находится в точке $x = 1$ (мы помним, что на смом деле в точке $x = 2$, но у нас есть ограничение $-\infty < x \le 1$, соответственно, минимум будет именно в точке $x = 1$). Однако, если мы используем ограничения...

In [ ]:
import numpy as np  # нам понадобится пакет numpy, чтобы использовать значение бесконечность (np.inf)
import math  #
from scipy.optimize import minimize  # берем готовую библиотечную функцию minimize
from scipy.optimize import LinearConstraint  # будем использовать линейные ограничения

In [ ]:
# описываем функцию, которую хотим минимизировать
def func_to_minimize(x):
    return (x - 2) * (x - 2) + 1 + 1 / (1 - x)

# Зададим начальное значение x
x = 0

# а теперь минимизируем нашу функцию
result = minimize(func_to_minimize, x)
print(result)  # выведем наш результа на экран

  message: Desired error not necessarily achieved due to precision loss.
  success: False
   status: 2
      fun: 6.0
        x: [ 0.000e+00]
      nit: 0
      jac: [-3.000e+00]
 hess_inv: [[1]]
     nfev: 174
     njev: 83


... то окажется, что минимум оказался в точке $0$. Не совсем тот результат, что мы ожидали.

Исправить положение можно либо изменив барьерную функцию так, чтобы она возрастала очень резко по мере приближения к границе запретной зоны, либо добавив специальный коэффициент $\alpha$, снизив влияние барьерной функции:

$$
f(x) = (x - 2)^2 + 1 + \alpha \cdot \frac {1} {1 - x}
$$

Попробуем:

In [ ]:
alpha = 0.1

# описываем функцию, которую хотим минимизировать
def func_to_minimize(x):
    return (x - 2) * (x - 2) + 1 + alpha * 1 / (1 - x)

# Зададим начальное значение x
x = 0

# а теперь минимизируем нашу функцию
result = minimize(func_to_minimize, x)
print(result)  # выведем наш результа на экран

  message: Desired error not necessarily achieved due to precision loss.
  success: False
   status: 2
      fun: 1.115695243467632
        x: [ 1.179e+00]
      nit: 2
      jac: [ 1.472e+00]
 hess_inv: [[ 1.595e-01]]
     nfev: 195
     njev: 95


Как видим, результат оказался гораздо лучше, хоть и неидеальным.

## Задание для самостоятельной работы

Изменить код в ячейке ниже (код основан на предыдущей работе) таким образом, чтобы учитывлось ограничение на время движения (необходимо затратить не более 0.028 часа), и расход топлива был минимальным, при условии использования штрафных или барьерных функций

In [1]:
import math
from scipy.optimize import minimize

h1 = 0.10
h2 = 0.10
l = 1

v1 = 40
v2 = 30

c1 = 0.115
c2 = 0.15

M = 1000  # коэффициент штрафа

def func_to_minimize(x):
    x = x[0]

    length1 = math.sqrt(h1*h1 + x*x)
    length2 = math.sqrt(h2*h2 + (l-x)*(l-x))

    # топливо (основная цель)
    total_fuel = length1 * c1 + length2 * c2

    # время (ограничение)
    total_time = length1 / v1 + length2 / v2

    # штраф
    penalty = max(0, total_time - 0.028)**2

    return total_fuel + M * penalty


x0 = [0]

bounds = [(0, l)]

result = minimize(func_to_minimize, x0, bounds=bounds)

print(result)
print("Оптимальное x:", result.x[0])


  message: CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL
  success: True
   status: 0
      fun: 0.1252808893957967
        x: [ 8.824e-01]
      nit: 5
      jac: [-8.477e-06]
     nfev: 14
     njev: 7
 hess_inv: <1x1 LbfgsInvHessProduct with dtype=float64>
Оптимальное x: 0.882386997831552
